# Raw IFS Frame Extraction — Stellar Spectrum

1. Generate a rectification matrix to perform the extraction.
2. Extract the lenslet spectra from the simulated raw frame.

## Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from liger_iris_sim.raw_ifs.rectmat import make_rectmat
from liger_iris_sim.raw_ifs.extraction import extract_ifs_lenslet

## Build rectification matrix

`make_rectmat` builds the per-lenslet linear map from raw detector pixels to trace-column flux — the extraction operator inverted by `extract_ifs_lenslet`. It only depends on the instrument mode/filter/resolution, not on any particular exposure, so it's cached to disk under `rectmats/` and reused across extractions rather than rebuilt every time.

In [ ]:
filter_name = "KN2"
resolution = 4000
rectmat_path = os.path.join("rectmats", f"rectmat_{filter_name}_{resolution}.fits")

if not os.path.exists(rectmat_path):
    make_rectmat(
        ifs_mode="lenslet",
        filter_name=filter_name,
        resolution=resolution,
        output_path=rectmat_path,
    )
    print(f"Built and cached rectification matrix at {rectmat_path}")
else:
    print(f"Using cached rectification matrix at {rectmat_path}")

## Load the simulated detector frame

Use the noiseless `SIM` frame and the high-resolution input spectrum (`WAVEHR`/`FLUXHR`) saved by `sim_raw_frame_liger_ifs`.

In [ ]:
input_path = "output/simulated_liger_raw_frame_star.fits"

with fits.open(input_path) as hdul:
    input_image = hdul["SIM"].data.astype(np.float32)
    wave_hr = hdul["WAVEHR"].data
    flux_hr = hdul["FLUXHR"].data

## Choose the output wavelength grid

Extraction resamples every lenslet onto a single common wavelength grid. Use the wavelength solution of the lenslet with the longest trace on the detector, which gives the widest wavelength coverage.

In [ ]:
with fits.open(rectmat_path) as hdul:
    offsets = hdul["OFFSETS"].data   # (2, n_lens_y, n_lens_x): [col_start, row_start]
    wavesol = hdul["WAVESOL"].data   # (max_trace_len, n_lens_y, n_lens_x), microns

from liger_iris_sim.utils import generate_wave_grid_for_filter
from liger_iris_drp_resources import load_filters_summary
filter_info = load_filters_summary('KN2')
wave_out = generate_wave_grid_for_filter(filter_info, resolution=resolution, sampling_factor=2)

## Extract the lenslet cube

`extract_ifs_lenslet` solves for each lenslet's flux column-by-column against the rectification matrix, then resamples every lenslet onto `wave_out`. The detector frame here is noiseless (`SIM`), so no `error` array is passed and an unweighted least-squares solve is used. `density=True` returns flux in phot/s/micron so it can be compared directly to the input spectrum.

In [ ]:
output_path = "output/extracted_cube_star.fits"

out = extract_ifs_lenslet(
    input_image, rectmat_path,
    error=None,
    output_wavesol=wave_out,
    interp_order=1,
    output_path=output_path,
)
cube_flux = out["flux"]   # (n_wave, n_lens_y, n_lens_x), phot/s
cube_err = out["err"]

print(f"Saved {output_path}")
print(f"  FLUX : {cube_flux.shape}")

## Compare the extracted spectrum to the input model

At lenslet (64, 64) — near the array center — compare the extracted spectrum to the high-resolution PHOENIX model that was broadcast to every lenslet in `sim_raw_frame_liger_ifs`.

In [ ]:
ly_c, lx_c = 55, 55
extracted = cube_flux[:, ly_c, lx_c].copy()
extracted /= np.gradient(wave_out)
plt.figure(figsize=(10, 4))
plt.plot(wave_hr, flux_hr, lw=1, label="Input PHOENIX model (high-res)")
plt.plot(wave_out, extracted, lw=1, label=f"Extracted lenslet ({ly_c}, {lx_c})")
plt.xlabel("Wavelength (μm)")
plt.ylabel("Flux (phot/s/μm)")
plt.title("Input vs. Extracted Spectrum")
plt.legend()
plt.tight_layout()
plt.show()

print(np.nanmean(extracted) / np.nanmean(flux_hr))